# Notebook 10 — 5-way baseline comparison (Louvain k=5)

**Goal:** Run all federated methods under the identical protocol on the Louvain k=5 split
(the paper's headline regime) and print one comparison table.

**Arms (same split, seeds 42–46, rounds=8, local_epochs=4, local_only_epochs=32, hidden=128):**
1. Local-only (computed inside every arm run)
2. Pooled (centralized, same per-hospital protocol — computed inside the FedAvg arm)
3. FedAvg
4. FedProx (μ=0.01) — proximal term on the client loss
5. FedMedian — coordinate-wise median server aggregation
6. FedTrimmedAvg (β=0.1) — drop top/bottom 10% of client updates, then average
7. Topology-aware (distinctiveness, T=1.0) — ours

Cells 6/7 are the notebook-09 k=5 arms (same code); cells 8–10 are the new baselines.
Run cells sequentially — each Ray simulation releases the GPU before the next starts.

**Do NOT re-tune anything after seeing results.** Any change = appended cell with date + rationale.

In [ ]:
# 1) Get the code + deps (record versions — deps are unpinned)
# Baselines live on a feature branch until merged to main — checkout after clone.
BRANCH = 'feat/federated-baselines'
!git clone -b main https://github.com/RawEgg6/Capstone-amr-fed.git 2>/dev/null || (cd Capstone-amr-fed && git fetch && git checkout main && git pull)
!cd /content/Capstone-amr-fed && git fetch origin {BRANCH} && git checkout {BRANCH} && git pull origin {BRANCH}
!cd /content/Capstone-amr-fed && git log -1 --oneline
!pip install -q torch_geometric 'flwr[simulation]' networkx
!pip freeze | grep -E '^(torch|torch-geometric|flwr|ray|networkx)==' || true

In [ ]:
# 2) Mount Drive + set ARMD_DIR
from google.colab import drive
drive.mount('/content/drive')
import os
os.environ['ARMD_DIR'] = '/content/drive/MyDrive/ARMD'   # edit to your folder if different

In [ ]:
# 3) Sanity: data resolves + imports
import sys
sys.path.insert(0, '/content/Capstone-amr-fed/src')
from amr_fed import config
from pathlib import Path
D = Path(config.DATA_DIR)
print('DATA_DIR:', D, '| exists:', D.exists())
assert D.exists(), 'ARMD_DIR is wrong — fix cell 2 and re-run.'
from amr_fed.federated.run import run_multiseed
from amr_fed.data_loader import load_cohort_frame, PK
from amr_fed.partition import louvain_split, partition_summary
print('imports OK')

In [ ]:
# 4) Load cohort ONCE — reused by every arm cell below
import sys as _s
for _m in [m for m in list(_s.modules) if m.startswith('amr_fed')]:
    del _s.modules[_m]
from amr_fed.data_loader import load_cohort_frame
df = load_cohort_frame()
print('rows:', f'{len(df):,}')

In [ ]:
# 5) SMOKE: validate the Louvain k=5 split before any training
import sys as _s
for _m in [m for m in list(_s.modules) if m.startswith('amr_fed')]:
    del _s.modules[_m]
from amr_fed.data_loader import load_cohort_frame, PK
from amr_fed.partition import louvain_split, partition_summary

print('\n--- Louvain k=5 ---')
assign = louvain_split(df, n_clients=5)
actual_k = assign.nunique()
g, niid = partition_summary(df, assign)
print(g)
print(f'non-IID(wstd): {niid:.4f} | actual hospitals: {actual_k}')
print(f'resist_rate spread: {g["resist_rate"].max() - g["resist_rate"].min():.4f}')
assert assign.index.is_unique and len(assign) == df[PK].nunique(), 'not 1:1'
assert (g['n_tests'] / g['n_tests'].sum() >= 0.01).all(), 'hospital below 1% size floor'
print(f'SMOKE PASS k=5')

print('\nSplit valid — safe to run arms.')

In [ ]:
# 6) LOUVAIN k=5, FedAvg arm (5 seeds) — same as nb 09 cell 6
# compute_pooled defaults to True, so this run also produces the pooled baseline.
import os, ray
os.environ.pop('AMR_RNG_PROBE', None)   # must NOT be set during arms
os.environ['AMR_DEBUG_INIT'] = '1'
ray.shutdown()
import sys as _s
for _m in [m for m in list(_s.modules) if m.startswith('amr_fed')]:
    del _s.modules[_m]
from amr_fed.federated.run import run_multiseed
from amr_fed.partition import louvain_split

lv5_fed = run_multiseed(
    seeds=(42, 43, 44, 45, 46), df=df,
    label='louvain-k5/FedAvg',
    partition_fn=lambda d, s: louvain_split(d, n_clients=5, seed=s),
    n_clients=5, rounds=8, local_epochs=4,
    local_only_epochs=32, hidden=128
)

In [ ]:
# 7) LOUVAIN k=5, topology arm (ours) — same as nb 09 cell 7
import os, ray
os.environ.pop('AMR_RNG_PROBE', None)
os.environ['AMR_DEBUG_INIT'] = '1'
ray.shutdown()
import sys as _s
for _m in [m for m in list(_s.modules) if m.startswith('amr_fed')]:
    del _s.modules[_m]
from amr_fed.federated.run import run_multiseed
from amr_fed.partition import louvain_split

lv5_topo = run_multiseed(
    seeds=(42, 43, 44, 45, 46), df=df,
    label='louvain-k5/TOPO',
    partition_fn=lambda d, s: louvain_split(d, n_clients=5, seed=s),
    n_clients=5, rounds=8, local_epochs=4,
    local_only_epochs=32, hidden=128,
    strategy='topology', topo_mode='distinctiveness', topo_temperature=1.0
)

In [ ]:
# 8) FedProx arm (μ=0.01) — proximal term penalises client drift from the global model
import os, ray
os.environ.pop('AMR_RNG_PROBE', None)
os.environ['AMR_DEBUG_INIT'] = '1'
ray.shutdown()
import sys as _s
for _m in [m for m in list(_s.modules) if m.startswith('amr_fed')]:
    del _s.modules[_m]
from amr_fed.federated.run import run_multiseed
from amr_fed.partition import louvain_split

lv5_fedprox = run_multiseed(
    seeds=(42, 43, 44, 45, 46), df=df,
    label='louvain-k5/FedProx',
    partition_fn=lambda d, s: louvain_split(d, n_clients=5, seed=s),
    n_clients=5, rounds=8, local_epochs=4,
    local_only_epochs=32, hidden=128,
    strategy='fedprox'
)

In [ ]:
# 9) FedMedian arm — coordinate-wise median server aggregation (outlier-robust)
import os, ray
os.environ.pop('AMR_RNG_PROBE', None)
os.environ['AMR_DEBUG_INIT'] = '1'
ray.shutdown()
import sys as _s
for _m in [m for m in list(_s.modules) if m.startswith('amr_fed')]:
    del _s.modules[_m]
from amr_fed.federated.run import run_multiseed
from amr_fed.partition import louvain_split

lv5_fedmedian = run_multiseed(
    seeds=(42, 43, 44, 45, 46), df=df,
    label='louvain-k5/FedMedian',
    partition_fn=lambda d, s: louvain_split(d, n_clients=5, seed=s),
    n_clients=5, rounds=8, local_epochs=4,
    local_only_epochs=32, hidden=128,
    strategy='fedmedian'
)

In [ ]:
# 10) FedTrimmedAvg arm — drop top/bottom β=10% of client updates, then average
import os, ray
os.environ.pop('AMR_RNG_PROBE', None)
os.environ['AMR_DEBUG_INIT'] = '1'
ray.shutdown()
import sys as _s
for _m in [m for m in list(_s.modules) if m.startswith('amr_fed')]:
    del _s.modules[_m]
from amr_fed.federated.run import run_multiseed
from amr_fed.partition import louvain_split

lv5_fedtrimmedavg = run_multiseed(
    seeds=(42, 43, 44, 45, 46), df=df,
    label='louvain-k5/FedTrimmedAvg',
    partition_fn=lambda d, s: louvain_split(d, n_clients=5, seed=s),
    n_clients=5, rounds=8, local_epochs=4,
    local_only_epochs=32, hidden=128,
    strategy='fedtrimmedavg'
)

In [ ]:
# 11) 5-way comparison table + verdict (Louvain k=5, seeds 42–46)
import json

def _fmt(ms):
    if not ms or ms[0] is None:
        return '—'
    return f"{ms[0]:.4f} ± {ms[1]:.4f}"

# Local-only / pooled come from the FedAvg arm run (computed there under the same protocol)
print(f"{'Method':<16} | {'Best-round F1 (mean±std)':<25} | {'Worst-hosp F1 (mean±std)':<25} | {'Final F1 (mean±std)':<20}")
print('-' * 98)
print(f"{'Local-only':<16} | {'—':<25} | {_fmt(lv5_fed['local_only']):<25} | {'—':<20}")
print(f"{'Pooled':<16} | {_fmt(lv5_fed['pooled']):<25} | {_fmt(lv5_fed['pooled_worst']):<25} | {'—':<20}")
for name, res in [('FedAvg', lv5_fed), ('FedProx(μ=0.01)', lv5_fedprox),
                  ('FedMedian', lv5_fedmedian), ('FedTrimmedAvg', lv5_fedtrimmedavg),
                  ('Topo-aware', lv5_topo)]:
    print(f"{name:<16} | {_fmt(res['fedavg_best']):<25} | {_fmt(res['worst_fed']):<25} | {_fmt(res['fedavg_final']):<20}")

# Worst-hospital deltas vs local-only (the fairness story)
wl = lv5_fed['worst_local'][0]
print('\nWorst-hospital delta vs local-only:')
for name, res in [('FedAvg', lv5_fed), ('FedProx(μ=0.01)', lv5_fedprox),
                  ('FedMedian', lv5_fedmedian), ('FedTrimmedAvg', lv5_fedtrimmedavg),
                  ('Topo-aware', lv5_topo)]:
    d = round(res['worst_fed'][0] - wl, 4)
    print(f'  {name:<16} {d:+.4f}   (worst_fed {res["worst_fed"][0]:.4f} vs local {wl:.4f})')

payload = {
    'split': 'louvain k=5', 'seeds': [42, 43, 44, 45, 46],
    'protocol': {'rounds': 8, 'local_epochs': 4, 'local_only_epochs': 32, 'hidden': 128},
    'local_only': lv5_fed['local_only'], 'worst_local': lv5_fed['worst_local'],
    'pooled': lv5_fed['pooled'], 'pooled_worst': lv5_fed['pooled_worst'],
    'fedavg':       {k: lv5_fed[k]       for k in ('fedavg_best', 'fedavg_final', 'worst_fed')},
    'fedprox':      {k: lv5_fedprox[k]   for k in ('fedavg_best', 'fedavg_final', 'worst_fed')},
    'fedmedian':    {k: lv5_fedmedian[k] for k in ('fedavg_best', 'fedavg_final', 'worst_fed')},
    'fedtrimmedavg':{k: lv5_fedtrimmedavg[k] for k in ('fedavg_best', 'fedavg_final', 'worst_fed')},
    'topology':     {k: lv5_topo[k]      for k in ('fedavg_best', 'fedavg_final', 'worst_fed')},
}
path = '/content/drive/MyDrive/amr_baseline_comparison.json'
with open(path, 'w') as f:
    json.dump(payload, f, default=str, indent=1)
print('saved', path)

print('\n=== COPY EVERYTHING BELOW THIS LINE BACK TO CHAT ===')
print(json.dumps(payload, default=str, indent=1))
print('=== END ===')